# 🤖 Notebook 3 — Pipeline ML Completo LEPI
**Habilidades demostradas:** Pipelines sklearn · Cross-validation · Gradient Boosting · Random Forest  

---
**Caso de negocio:** Predecir el ILEPI futuro de un docente y detectar
quiénes están en riesgo de bajo desempeño antes de la evaluación formal.


In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
from data_generator import generar_docentes, generar_impacto, resumen_institucional
df = generar_docentes(300)
df_impacto = generar_impacto(15)
print(f"✅ Datos cargados — {len(df)} docentes | {len(df_impacto)} instituciones")
df.head()


## 1. Pipeline sklearn — Predicción de ILEPI

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, roc_auc_score, roc_curve
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

FEATURES = ['anos_exp','edad','lid_vision','lid_comunicacion','prax_ver',
            'prax_juzgar','inn_tecnologia','satisfaccion','carga_laboral',
            'ratio_form_exp','horas_formacion','evaluacion_par']

X = df[FEATURES]; y = df['ILEPI']
X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.2,random_state=42)

pipe_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  GradientBoostingRegressor(n_estimators=200,learning_rate=0.05,
                                          max_depth=3,subsample=0.8,random_state=42))
])
pipe_reg.fit(X_tr, y_tr)
y_pred = pipe_reg.predict(X_te)
r2     = r2_score(y_te, y_pred)
mae    = mean_absolute_error(y_te, y_pred)
cv     = cross_val_score(pipe_reg, X, y, cv=5, scoring='r2')

print(f"R²  (test)          : {r2:.4f}  {'✅' if r2>0.75 else '⚠️'}")
print(f"MAE (test)          : {mae:.4f}")
print(f"R²  (CV 5-fold)     : {cv.mean():.4f} ± {cv.std():.4f}")
print(f"Sobreajuste (R²-CV) : {r2 - cv.mean():.4f}  {'✅ Bajo' if abs(r2-cv.mean())<0.05 else '⚠️ Revisar'}")


## 2. Importancia de Variables + SHAP manual

In [ ]:
imp = pd.Series(
    pipe_reg.named_steps['model'].feature_importances_,
    index=FEATURES
).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(imp.index, imp.values,
             color=plt.cm.Blues(np.linspace(0.2, 1.0, len(imp))))
axes[0].set_title('Importancia de Variables\n(Predicción ILEPI)')
axes[0].set_xlabel('Feature Importance')

residuos = y_te.values - y_pred
axes[1].scatter(y_pred, residuos, alpha=0.5, color='steelblue', s=25)
axes[1].axhline(0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('ILEPI Predicho'); axes[1].set_ylabel('Residuo')
axes[1].set_title('Gráfico de Residuos\n(Heterocedasticidad)')

plt.suptitle('Diagnóstico del Modelo — Gradient Boosting LEPI', fontweight='bold')
plt.tight_layout(); plt.show()

print(f"\nTop 5 variables predictoras:")
print(imp.tail(5).sort_values(ascending=False).round(4))


## 3. Clasificación — Alerta de Riesgo Docente

In [ ]:
y_riesgo = (df['ILEPI'] < 3.0).astype(int)
FEATURES_CLF = FEATURES + ['ILD','IRPD','IIP']
X2  = df[FEATURES_CLF]
X2_tr,X2_te,y2_tr,y2_te = train_test_split(X2,y_riesgo,test_size=0.2,random_state=42,stratify=y_riesgo)

pipe_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  RandomForestClassifier(n_estimators=200,class_weight='balanced',
                                       max_depth=6,random_state=42))
])
pipe_clf.fit(X2_tr, y2_tr)
y_prob2 = pipe_clf.predict_proba(X2_te)[:,1]
y_pred2 = pipe_clf.predict(X2_te)
auc     = roc_auc_score(y2_te, y_prob2)

print(f"AUC-ROC: {auc:.4f}  {'✅' if auc>0.80 else '⚠️'}")
print(classification_report(y2_te, y_pred2, target_names=['Desempeño OK','Riesgo bajo']))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fpr,tpr,_ = roc_curve(y2_te,y_prob2)
axes[0].plot(fpr,tpr,'g-',lw=2,label=f'AUC={auc:.3f}')
axes[0].plot([0,1],[0,1],'r--'); axes[0].fill_between(fpr,tpr,alpha=0.1,color='green')
axes[0].set_title('Curva ROC — Alerta LEPI'); axes[0].legend()

imp2 = pd.Series(pipe_clf.named_steps['model'].feature_importances_,
                 index=FEATURES_CLF).sort_values().tail(8)
axes[1].barh(imp2.index, imp2.values, color=plt.cm.Greens(np.linspace(0.4,1.0,8)))
axes[1].set_title('Variables Predictoras\nRiesgo Bajo Desempeño')

ConfusionMatrixDisplay.from_predictions(y2_te,y_pred2,
    display_labels=['OK','Riesgo'],ax=axes[2],colorbar=False,cmap='Greens')
axes[2].set_title('Matriz de Confusión')

plt.suptitle('Pipeline Clasificación — App LEPI', fontweight='bold')
plt.tight_layout(); plt.show()


## 4. Predicción Docente Individual

In [ ]:
def predecir_docente_nuevo(datos_doc: dict) -> None:
    """Sistema de predicción para un docente nuevo en la app LEPI."""
    df_new = pd.DataFrame([datos_doc])
    ilepi  = pipe_reg.predict(df_new[FEATURES])[0]
    prob_r = pipe_clf.predict_proba(df_new[FEATURES_CLF])[0][1]
    alerta = ('🔴 Riesgo Alto'  if prob_r > 0.7 else
              '🟡 Riesgo Medio' if prob_r > 0.4 else '🟢 Óptimo')
    print(f"  ILEPI Predicho  : {ilepi:.3f}")
    print(f"  Prob. Riesgo    : {prob_r:.1%}")
    print(f"  Alerta          : {alerta}")

# Docente en riesgo
print("═"*40); print("Docente A (indicadores bajos):")
predecir_docente_nuevo({
    'anos_exp':2,'edad':27,'lid_vision':2.5,'lid_comunicacion':2.8,
    'prax_ver':2.4,'prax_juzgar':2.2,'inn_tecnologia':2.3,
    'satisfaccion':3.0,'carga_laboral':4.5,'ratio_form_exp':0.5,
    'horas_formacion':10,'evaluacion_par':2.5,
    'ILD':2.6,'IRPD':2.4,'IIP':2.3
})
# Docente avanzado
print("═"*40); print("Docente B (indicadores altos):")
predecir_docente_nuevo({
    'anos_exp':15,'edad':42,'lid_vision':4.5,'lid_comunicacion':4.3,
    'prax_ver':4.2,'prax_juzgar':4.0,'inn_tecnologia':4.4,
    'satisfaccion':4.5,'carga_laboral':3.0,'ratio_form_exp':6.2,
    'horas_formacion':95,'evaluacion_par':4.4,
    'ILD':4.3,'IRPD':4.1,'IIP':4.2
})


## ✅ Conclusiones Técnicas

### Habilidades demostradas
- **sklearn Pipelines**: Encadenamiento scaler + modelo, reutilizable y limpio
- **Gradient Boosting**: Tuning de hiperparámetros, control del sobreajuste
- **Cross-validation**: Validación robusta con 5 folds
- **Diagnóstico de modelos**: Curva ROC, residuos, importancia de variables
- **Sistema de alertas**: Predicción individual lista para producción
